# 優先度の高い実験（スタッキング・3本ブレンド・既存提出ブレンド）

**参照**: `docs/18_CURRENT_STATUS_AND_ROADMAP.md` の優先度の高い項目を実行する。

**方針**: ベースは **bpr64_count1 + BPR128**。ブレンド比率は探索済みで **0.65 : 0.35**（count1 : BPR128）が最良のため、この比率を全ブレンドに適用する。

| # | 内容 | 提出ファイル例 |
|---|------|----------------|
| 2 | スタッキング 1 本 + BPR128 ブレンド（0.65:0.35） | submission_blend_stacking_bpr128_w065.csv |
| 3 | 3 本ブレンド（count1 + BPR128 + stacking） | submission_blend_3way_equal.csv 等 |
| 4 | 既存の良提出を BPR128 と 0.65:0.35 でブレンド | submission_blend_similar_bpr128_w065.csv 等 |
| 5 | 3 本ブレンドの重みバリエーション | submission_blend_3way_050_025_025.csv 等 |

## セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import get_setup, get_bpr_base, run_03_stacking
from lib.submission import (
    save_submission,
    verify_submission,
    blend_two_submissions,
    blend_n_submissions,
)
from lib.two_hop import add_2hop_features, TWO_HOP_REVIEW_COUNT, run_experiment

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")

# 最高精度の左側: bpr64_count1（55 + BPR64 + movie_review_count）
path_count1 = ctx.submissions_dir / "submission_2hop_bpr64_count1.csv"
if not path_count1.exists():
    r = run_experiment(ctx, "bpr64_count1", bpr_factors=64, use_2hop_cols=[TWO_HOP_REVIEW_COUNT])
    print(f"  [bpr64_count1] → {path_count1.name}  ({'OK' if r.get('ok') else r.get('message')})")
else:
    print(f"  [bpr64_count1] 既存: {path_count1.name}")

# 最高精度の右側: BPR128
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
if not path_bpr128.exists():
    r = run_experiment(ctx, "bpr128_only", bpr_factors=128)
    print(f"  [BPR128] → {path_bpr128.name}  ({'OK' if r.get('ok') else r.get('message')})")
else:
    print(f"  [BPR128] 既存: {path_bpr128.name}")

test_ids = ctx.test["ID"].values
# 最良比率（0.65:0.35）で参照用ベスト提出を常に更新
path_best = ctx.submissions_dir / "submission_blend_bpr64_count1_bpr128.csv"
blend_two_submissions(path_count1, path_bpr128, path_best, weight_a=0.65, test_ids=test_ids)
print(f"  参照: {path_best.name} (0.65:0.35)")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
  [bpr64_count1] 既存: submission_2hop_bpr64_count1.csv
  [BPR128] 既存: submission_2hop_bpr128_only.csv
  参照: submission_blend_bpr64_count1_bpr128.csv (0.65:0.35)
提出先: outputs/submissions


## 1. ブレンド比率（確定済み・スキップ）

探索の結果 **0.65 : 0.35**（count1 : BPR128）が最良。セットアップで `submission_blend_bpr64_count1_bpr128.csv` をこの比率で作成済み。

In [2]:
# 比率探索は完了済み。最良 0.65:0.35 はセットアップで path_best に反映済み。
print("  スキップ: ブレンド比率は 0.65:0.35 に固定済み")

  スキップ: ブレンド比率は 0.65:0.35 に固定済み


## 2. スタッキング + BPR128 ブレンド（0.65:0.35）

BPR64 土台のスタッキング 1 本と BPR128 を最良比率 0.65:0.35 でブレンド。

In [3]:
# bpr64_count1 の ctx を用意（スタッキング用）
train_c1, test_c1, feats_c1 = get_bpr_base(ctx, factors=64)
add_2hop_features(train_c1, test_c1, columns=[TWO_HOP_REVIEW_COUNT])
feats_c1 = feats_c1 + [TWO_HOP_REVIEW_COUNT]
for col in feats_c1:
    if not pd.api.types.is_numeric_dtype(train_c1[col]) and not pd.api.types.is_categorical_dtype(train_c1[col]):
        train_c1[col] = train_c1[col].astype("category")
        test_c1[col] = test_c1[col].astype("category")
ctx_bpr_count1 = ctx._replace(
    train=train_c1, test=test_c1,
    X=train_c1[feats_c1], X_test=test_c1[feats_c1], features=feats_c1,
)

path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"
if not path_stacking.exists():
    r = run_03_stacking(ctx_bpr_count1, seed=None)
    if r.get("path"):
        path_stacking = Path(r["path"])
        print(f"  [スタッキング] → {path_stacking.name}  ({'OK' if r.get('ok') else r.get('message')})")
    else:
        print(f"  [スタッキング] スキップ: {r.get('message')}")
else:
    print(f"  [スタッキング] 既存: {path_stacking.name}")

if path_stacking.exists():
    out_name = "submission_blend_stacking_bpr128_w065.csv"
    out_path = ctx.submissions_dir / out_name
    r = blend_two_submissions(
        path_stacking, path_bpr128, out_path,
        weight_a=0.65, test_ids=test_ids,
    )
    status = "OK" if r.get("ok") else r.get("message", "")
    print(f"  stacking:BPR128=0.65:0.35 → {out_name}  ({status})")

  0%|          | 0/100 [00:00<?, ?it/s]

  [スタッキング] → submission_improvement_03_stacking.csv  (OK)
  stacking:BPR128=0.65:0.35 → submission_blend_stacking_bpr128_w065.csv  (OK)


## 3. 3 本ブレンド（count1 + BPR128 + stacking）

均等 (1/3, 1/3, 1/3) と count1 やや多め (0.4, 0.3, 0.3) を試す。

In [4]:
path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"
if path_stacking.exists():
    configs_3way = [
        ([path_count1, path_bpr128, path_stacking], [1, 1, 1], "submission_blend_3way_equal.csv"),
        ([path_count1, path_bpr128, path_stacking], [0.4, 0.3, 0.3], "submission_blend_3way_040_030_030.csv"),
    ]
    for paths, weights, out_name in configs_3way:
        out_path = ctx.submissions_dir / out_name
        r = blend_n_submissions(paths, weights, out_path, test_ids=test_ids)
        status = "OK" if r.get("ok") else r.get("message", "")
        print(f"  {out_name}  ({status})")
else:
    print("  [3本ブレンド] スキップ: submission_improvement_03_stacking.csv がありません")

  submission_blend_3way_equal.csv  (OK)
  submission_blend_3way_040_030_030.csv  (OK)


## 追加候補（このノートで試せるもの）

- **2 本ブレンド**: すべて 0.65:0.35（左: 右）で統一済み。
- **3 本ブレンドの重みバリエーション**: (0.5, 0.25, 0.25)、(0.25, 0.5, 0.25)、(0.25, 0.25, 0.5) など（上記 §5）。
- **スタッキング複数 seed**: `run_03_stacking_batch` で seed 42〜46 の 5 本を出し、それぞれ BPR128 と 0.65:0.35 でブレンド（要・学習時間）。
- **4 本ブレンド**: count1 + BPR128 + stacking + similar_movies（または improvement_05）を均等 or 重み付き（既存 CSV があれば CSV だけで可）。

## 4. 既存の良提出を BPR128 と 0.65:0.35 でブレンド

similar_movies_reviewed・improvement_05 の CSV が既にあれば、BPR128 と最良比率 0.65:0.35 でブレンド。学習は不要。

In [5]:
# 既存の良提出と BPR128 を 0.65:0.35 でブレンド
candidates = [
    ("submission_similar_movies_reviewed.csv", "similar"),
    ("submission_improvement_05_scale_pos_weight.csv", "improvement05"),
]
for csv_name, prefix in candidates:
    path_left = ctx.submissions_dir / csv_name
    if not path_left.exists():
        print(f"  [{prefix}] スキップ: {csv_name} がありません")
        continue
    out_name = f"submission_blend_{prefix}_bpr128_w065.csv"
    out_path = ctx.submissions_dir / out_name
    r = blend_two_submissions(path_left, path_bpr128, out_path, weight_a=0.65, test_ids=test_ids)
    status = "OK" if r.get("ok") else r.get("message", "")
    print(f"  [{prefix}] 0.65:0.35 → {out_name}  ({status})")

  [similar] スキップ: submission_similar_movies_reviewed.csv がありません
  [improvement05] スキップ: submission_improvement_05_scale_pos_weight.csv がありません


## 5. 3 本ブレンドの重みバリエーション

count1 / BPR128 / stacking の 3 本で、どれを多めにするか変えたパターンを試す。

In [6]:
path_stacking = ctx.submissions_dir / "submission_improvement_03_stacking.csv"
if path_stacking.exists():
    configs = [
        ([path_count1, path_bpr128, path_stacking], [0.5, 0.25, 0.25], "submission_blend_3way_050_025_025.csv"),
        ([path_count1, path_bpr128, path_stacking], [0.25, 0.5, 0.25], "submission_blend_3way_025_050_025.csv"),
        ([path_count1, path_bpr128, path_stacking], [0.25, 0.25, 0.5], "submission_blend_3way_025_025_050.csv"),
    ]
    for paths, weights, out_name in configs:
        out_path = ctx.submissions_dir / out_name
        r = blend_n_submissions(paths, weights, out_path, test_ids=test_ids)
        print(f"  {out_name}  ({r.get('message', '')})")
else:
    print("  [3本ブレンド 重み] スキップ: submission_improvement_03_stacking.csv がありません")

  submission_blend_3way_050_025_025.csv  (OK)
  submission_blend_3way_025_050_025.csv  (OK)
  submission_blend_3way_025_025_050.csv  (OK)


## 6. 比率の細かめグリッド（スキップ済み）

探索の結果 0.65:0.35 に確定済みのため実施しない。

In [7]:
# 比率は 0.65 に確定済みのためスキップ
print("  スキップ: 細かめグリッドは不要")

  スキップ: 細かめグリッドは不要


## 提出ファイル一覧

In [8]:
sub_dir = Path(ctx.submissions_dir)
patterns = [
    "submission_blend_stacking_bpr128_",
    "submission_blend_3way_",
    "submission_blend_similar_bpr128_",
    "submission_blend_improvement05_bpr128_",
]
listed = []
for p in sorted(sub_dir.glob("*.csv")):
    if p.name == "submission_blend_bpr64_count1_bpr128.csv" or any(p.name.startswith(pat) for pat in patterns):
        listed.append(p.name)
for name in sorted(set(listed)):
    p = sub_dir / name
    v = verify_submission(p, ctx.test)
    print(f"  {name}  ({v.get('message', '')})")

  submission_blend_3way_025_025_050.csv  (OK)
  submission_blend_3way_025_050_025.csv  (OK)
  submission_blend_3way_040_030_030.csv  (OK)
  submission_blend_3way_050_025_025.csv  (OK)
  submission_blend_3way_equal.csv  (OK)
  submission_blend_bpr64_count1_bpr128.csv  (OK)
  submission_blend_stacking_bpr128_w065.csv  (OK)
